In [ ]:
%pip install -U -q transformers datasets accelerate scikit-learn "protobuf==3.20.3" sentencepiece

In [ ]:
import numpy as np
import torch
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    TrainerCallback
)
import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = 'cuda'

In [ ]:
ds = load_dataset("ailsntua/QEvasion")

train_full_df = ds['train'].to_pandas()
test_df = ds['test'].to_pandas()

train_df, val_df = train_test_split(
    train_full_df,
    test_size=0.2,
    random_state=SEED,
    stratify=train_full_df['clarity_label']
)

print(f"Train: {len(train_df)}")
print(f"Validation: {len(val_df)}")
print(f"Test: {len(test_df)}")

In [ ]:
MODEL_NAME = "answerdotai/ModernBERT-large"
MAX_LENGTH = 3000

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

label2id = {"Clear Reply": 0, "Ambivalent": 1, "Clear Non-Reply": 2}
id2label = {v: k for k, v in label2id.items()}
clarity_labels = ["Clear Reply", "Ambivalent", "Clear Non-Reply"]

In [ ]:
def tokenize_function(examples):
    inputs = [
        f"Question: {q}\nAnswer: {a}"
        for q, a in zip(examples["question"], examples["interview_answer"])
    ]
    tokenized = tokenizer(
        inputs,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    )
    tokenized["labels"] = [label2id[l] for l in examples["clarity_label"]]
    return tokenized

train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
val_dataset = Dataset.from_pandas(val_df, preserve_index=False)

train_tokenized = train_dataset.map(tokenize_function, batched=True, remove_columns=train_dataset.column_names)
val_tokenized = val_dataset.map(tokenize_function, batched=True, remove_columns=val_dataset.column_names)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='macro')
    return {'accuracy': accuracy, 'f1': f1}

In [ ]:
class EarlyStoppingCallback(TrainerCallback):
    def __init__(self, patience: int = 3, metric_name: str = "eval_f1", greater_is_better: bool = True):
        self.patience = patience
        self.metric_name = metric_name
        self.greater_is_better = greater_is_better
        self.best_metric = None
        self.patience_counter = 0
        
    def on_evaluate(self, args, state, control, metrics, **kwargs):
        current_metric = metrics.get(self.metric_name)
        
        if current_metric is None:
            return
        
        if self.best_metric is None:
            self.best_metric = current_metric
            self.patience_counter = 0
        else:
            if self.greater_is_better:
                improved = current_metric > self.best_metric
            else:
                improved = current_metric < self.best_metric
            
            if improved:
                self.best_metric = current_metric
                self.patience_counter = 0
            else:
                self.patience_counter += 1
                
        if self.patience_counter >= self.patience:
            print(f"\nEarly stopping triggered!")
            print(f"Best {self.metric_name}: {self.best_metric:.4f}")
            control.should_training_stop = True

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
    reference_compile=False,  # Disable torch.compile to avoid CUDA errors with gradient checkpointing
    attn_implementation="sdpa",  # Use PyTorch 2.0 scaled dot-product attention (optimal for RTX 3090)
    trust_remote_code=True
)

In [ ]:
training_args = TrainingArguments(
    output_dir="./results_modernbert_simple",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    warmup_ratio=0.05,
    weight_decay=0.10,
    max_grad_norm=1.0,
    gradient_checkpointing=True,
    bf16=True,
    bf16_full_eval=True,
    optim="adamw_torch_fused",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    logging_strategy="steps",
    seed=SEED,
    report_to="none",
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(patience=3, metric_name="eval_f1", greater_is_better=True)],
)

In [ ]:

trainer.train()

In [ ]:
def tokenize_test(examples):
    texts = [
        f"Question: {q}\nAnswer: {a}"
        for q, a in zip(examples['question'], examples['interview_answer'])
    ]
    return tokenizer(texts, truncation=True, padding=False, max_length=MAX_LENGTH)

test_dataset = Dataset.from_pandas(test_df, preserve_index=False)
test_tokenized = test_dataset.map(
    tokenize_test, 
    batched=True, 
    remove_columns=[col for col in test_dataset.column_names if col not in ['clarity_label']]
)

predictions_output = trainer.predict(test_tokenized)
predictions = np.argmax(predictions_output.predictions, axis=-1)
predicted_labels = [id2label[pred] for pred in predictions]

y_true = test_df['clarity_label'].tolist()
y_pred = predicted_labels

accuracy = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average='macro', labels=clarity_labels, zero_division=0)
weighted_f1 = f1_score(y_true, y_pred, average='weighted', labels=clarity_labels, zero_division=0)
per_class_f1 = f1_score(y_true, y_pred, average=None, labels=clarity_labels, zero_division=0)

print(f"\nAccuracy: {accuracy:.4f}")
print(f"Macro F1: {macro_f1:.4f}")
print(f"Weighted F1: {weighted_f1:.4f}")

print(f"\nPer-class F1 scores:")
for label, f1_val in zip(clarity_labels, per_class_f1):
    print(f"  {label}: {f1_val:.4f}")

print("\nClassification Report:")
print(classification_report(y_true, y_pred, labels=clarity_labels, zero_division=0))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_true, y_pred, labels=clarity_labels)
print(f"Labels: {clarity_labels}")
print(cm)